In [2]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.2 MB/s eta 0:00:0000:01


# READ DATASET

In [3]:
import numpy as np
import pandas as pd
import torch
from torch_geometric.datasets.movie_lens_1m import MovieLens1M

torch.manual_seed(42)
np.random.seed(42)


dataset = MovieLens1M(root="./data/MovieLens1M")
data = dataset[0]

num_users_raw = data["user"].num_nodes
num_movies_raw = data["movie"].num_nodes

edge_index = data["user", "rates", "movie"].edge_index
ratings = data["user", "rates", "movie"].rating.float()
timestamps = data["user", "rates", "movie"].time.long()

print("=" * 60)
print("RAW DATASET")
print("=" * 60)
print(f"Users        : {num_users_raw}")
print(f"Movies       : {num_movies_raw}")
print(f"Interactions : {edge_index.size(1)}")

Extracting data/MovieLens1M/ml-1m.zip
Processing...


RAW DATASET
Users        : 6040
Movies       : 3883
Interactions : 1000209


Done!


In [4]:
df = pd.DataFrame({
    "user": edge_index[0].numpy(),
    "movie": edge_index[1].numpy(),
    "rating": ratings.numpy(),
    "timestamp": timestamps.numpy()
})


print("\nRating Distribution")
print(df["rating"].value_counts().sort_index())

user_interactions = df.groupby("user").size()
movie_interactions = df.groupby("movie").size()

print("\nInteraction Statistics")
print(f"User   : min={user_interactions.min()} "
      f"median={user_interactions.median()} "
      f"max={user_interactions.max()}")

print(f"Movie  : min={movie_interactions.min()} "
      f"median={movie_interactions.median()} "
      f"max={movie_interactions.max()}")


Rating Distribution
rating
1.0     56174
2.0    107557
3.0    261197
4.0    348971
5.0    226310
Name: count, dtype: int64

Interaction Statistics
User   : min=20 median=96.0 max=2314
Movie  : min=1 median=123.5 max=3428


# Convert Explicit -> Implicit Feedback

In [5]:
POS_THRESHOLD = 4.0

df = df[df["rating"] >= POS_THRESHOLD].copy()

print("\nImplicit Feedback")
print(f"Positive interactions: {len(df)}")


Implicit Feedback
Positive interactions: 575281


# Remove users with fewer than 5 positive interactions

In [6]:
MIN_INTERACTIONS = 5

user_counts = df.groupby("user").size()

valid_users = user_counts[user_counts >= MIN_INTERACTIONS].index

df = df[df["user"].isin(valid_users)].copy()

print(f"Remaining users : {df['user'].nunique()}")
print(f"Remaining movies: {df['movie'].nunique()}")

Remaining users : 6034
Remaining movies: 3533


# Re-index User and Movie IDs

In [7]:
unique_users = np.sort(df["user"].unique())
unique_movies = np.sort(df["movie"].unique())

user2idx = {u: i for i, u in enumerate(unique_users)}
movie2idx = {m: i for i, m in enumerate(unique_movies)}

df["user_idx"] = df["user"].map(user2idx)
df["movie_idx"] = df["movie"].map(movie2idx)

num_users = len(unique_users)
num_items = len(unique_movies)

print("\nAfter Re-index")
print(f"Users : {num_users}")
print(f"Items : {num_items}")


After Re-index
Users : 6034
Items : 3533


# SPLIT TRAIN/VAL/TEST

In [8]:
def split_per_user(group):

    group = group.sort_values("timestamp").copy()

    group["split"] = "train"

    group.iloc[-2, group.columns.get_loc("split")] = "val"
    group.iloc[-1, group.columns.get_loc("split")] = "test"

    return group


df = (
    df.groupby("user_idx", group_keys=False)
      .apply(split_per_user)
      .reset_index(drop=True)
)

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

print("\nDataset Split")
print(df["split"].value_counts())


Dataset Split
split
train    563204
val        6034
test       6034
Name: count, dtype: int64


/tmp/ipykernel_58/2764612157.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(split_per_user)


In [9]:
train_edges = torch.tensor(
    train_df[["user_idx", "movie_idx"]].values,
    dtype=torch.long
)

val_edges = torch.tensor(
    val_df[["user_idx", "movie_idx"]].values,
    dtype=torch.long
)

test_edges = torch.tensor(
    test_df[["user_idx", "movie_idx"]].values,
    dtype=torch.long
)

In [ ]:
# =========================================================
# Evaluation lookup structures
# =========================================================
# train_user_pos_items[u] is the set of remapped item IDs that user u
# interacted with in the TRAIN split only. Validation/test items are excluded.
train_user_pos_items = [set() for _ in range(num_users)]

for user_id, item_id in train_edges.tolist():
    train_user_pos_items[user_id].add(item_id)


# val_items[u] is the single remapped validation item ID for user u.
# A value of -1 means missing, but each retained user should have one item.
val_items = torch.full((num_users,), -1, dtype=torch.long)
val_items[val_edges[:, 0]] = val_edges[:, 1]


# test_items[u] is the single remapped test item ID for user u.
# A value of -1 means missing, but each retained user should have one item.
test_items = torch.full((num_users,), -1, dtype=torch.long)
test_items[test_edges[:, 0]] = test_edges[:, 1]


assert len(train_user_pos_items) == num_users
assert (val_items >= 0).sum().item() == num_users
assert (test_items >= 0).sum().item() == num_users

# Build Bipartite Graph

In [10]:
movie_offset = num_users

user_nodes = train_edges[:, 0]
movie_nodes = train_edges[:, 1] + movie_offset

src = torch.cat([user_nodes, movie_nodes])
dst = torch.cat([movie_nodes, user_nodes])

edge_index_train = torch.stack([src, dst], dim=0)

print("\nGraph")
print(f"edge_index_train shape: {tuple(edge_index_train.shape)}")


Graph
edge_index_train shape: (2, 1126408)


# Item degree: computed ONLY from training graph

In [11]:
item_degree = (
    train_df
    .groupby("movie_idx")
    .size()
    .reindex(range(num_items), fill_value=0)
)

item_degree = torch.tensor(
    item_degree.values,
    dtype=torch.long
)

# Split item to head/middle/tail
Tail   : bottom 50% <br>
Middle : 30% <br>
Head   : top 20% <br>

Encoding: <br>

0 = Tail <br>
1 = Middle <br>
2 = Head <br>

In [ ]:
deg_np = item_degree.numpy()

p50 = np.percentile(deg_np, 50)
p80 = np.percentile(deg_np, 80)

item_popularity_group = torch.zeros(
    num_items,
    dtype=torch.long
)

item_popularity_group[deg_np >= p50] = 1
item_popularity_group[deg_np >= p80] = 2

print("\nPopularity Groups")
print(f"Tail   : {(item_popularity_group == 0).sum().item()}")
print(f"Middle : {(item_popularity_group == 1).sum().item()}")
print(f"Head   : {(item_popularity_group == 2).sum().item()}")




Popularity Groups
Tail   : 691
Middle : 2135
Head   : 707


# Statistics

In [ ]:
dataset_statistics = pd.DataFrame({

    "Metric": [
        "Users",
        "Items",
        "Positive interactions",
        "Train interactions",
        "Validation interactions",
        "Test interactions",
        "Average interactions / user",
        "Average interactions / item"
    ],

    "Value": [
        num_users,
        num_items,
        len(df),
        len(train_df),
        len(val_df),
        len(test_df),
        round(len(df) / num_users, 2),
        round(len(df) / num_items, 2)
    ]

})

print("\nDataset Statistics")
print(dataset_statistics)

# =========================================================
# Final output sanity checks
# =========================================================
print("\nSanity Checks")
print(f"number of users                  : {num_users}")
print(f"number of items                  : {num_items}")
print(f"train_edges shape                : {tuple(train_edges.shape)}")
print(f"val_edges shape                  : {tuple(val_edges.shape)}")
print(f"test_edges shape                 : {tuple(test_edges.shape)}")
print(f"edge_index_train shape           : {tuple(edge_index_train.shape)}")
print(f"item_degree shape                : {tuple(item_degree.shape)}")
print(f"item_popularity_group shape      : {tuple(item_popularity_group.shape)}")
print(f"users in train_user_pos_items    : {len(train_user_pos_items)}")
print(f"validation users                 : {(val_items >= 0).sum().item()}")
print(f"test users                       : {(test_items >= 0).sum().item()}")

# =========================================================
# Final objects to hand over
# =========================================================
outputs = {

    "train_edges": train_edges,

    "val_edges": val_edges,

    "test_edges": test_edges,

    "edge_index_train": edge_index_train,

    "num_users": num_users,

    "num_items": num_items,

    "item_degree": item_degree,

    "item_popularity_group": item_popularity_group,

    "train_user_pos_items": train_user_pos_items,

    "val_items": val_items,

    "test_items": test_items,

    "dataset_statistics": dataset_statistics
}

print("\nPreprocessing completed successfully!")

# Evaluation Pipeline Test: MostPopular Baseline

In [ ]:
from pathlib import Path
import sys

# Allow imports from src/ when this notebook is executed from notebooks/.
project_root = Path.cwd().resolve()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.baselines import MostPopular
from src.metrics import evaluate_full_ranking


# MostPopular assigns every user the same training-set item popularity scores.
# It is used here only as a sanity check for the full-ranking evaluation
# pipeline, including training-item masking inside evaluate_full_ranking.
mostpopular = MostPopular().fit(train_edges, num_items)
mostpopular_scores = mostpopular.predict_all(num_users)

mostpopular_results = evaluate_full_ranking(
    scores=mostpopular_scores,
    train_user_pos_items=train_user_pos_items,
    test_items=test_items,
    item_group=item_popularity_group,
    item_degree=item_degree,
    k_list=[10, 20],
)

print("\nMostPopular result dictionary")
print(mostpopular_results)

print("\nMostPopular metrics")
for metric_name, metric_value in mostpopular_results.items():
    print(f"{metric_name}: {metric_value:.6f}")

metrics_dir = project_root / "results" / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)

mostpopular_results_df = pd.DataFrame([mostpopular_results])
mostpopular_results_path = metrics_dir / "mostpopular_results.csv"
mostpopular_results_df.to_csv(mostpopular_results_path, index=False)

print(f"\nSaved MostPopular results to: {mostpopular_results_path}")